In [ ]:
import hashlib
from hash_src.merkle_tree import MerkleTree
from hash_src.rescue import RescuePrime

In [17]:
import hashlib

# ==========================================
# 1. DEFINE HASH VARIANTS
# ==========================================
def sha256_standard(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

def sha256_double(data: bytes) -> bytes:
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()

def sha256_salted(data: bytes) -> bytes:
    salt = b"Secure_Salt_2026_"
    return hashlib.sha256(salt + data).digest()

hash_variants = {
    "Standard SHA-256": sha256_standard,
    "Double SHA-256 (Bitcoin style)": sha256_double,
    "Salted SHA-256": sha256_salted
}

# ==========================================
# 2. DETAILED MULTI-LAYER MERKLE TREE
# ==========================================
class MerkleTree:
    def __init__(self, leaves: list[bytes], hash_fn):
        self.hash_fn = hash_fn
        # Hash initial leaf transactions
        self.leaves = [self.hash_fn(leaf) for leaf in leaves]
        self.layers = [self.leaves]
        self.build_tree()
        
    @property
    def root(self) -> bytes:
        return self.layers[-1][0] if self.layers[-1] else b""

    def build_tree(self):
        current_layer = self.leaves
        layer_idx = 0
        
        while len(current_layer) > 1:
            print(f"  [Layer {layer_idx} -> Layer {layer_idx + 1} Processing]")
            next_layer = []
            
            for i in range(0, len(current_layer), 2):
                left = current_layer[i]
                
                # Check for odd-numbered child element to duplicate
                if i + 1 < len(current_layer):
                    right = current_layer[i+1]
                    print(f"    - Hashing children pairs: {left.hex()[:8]}... + {right.hex()[:8]}...")
                else:
                    right = left
                    print(f"    - Odd child duplication: {left.hex()[:8]}... + (Duplicated) {right.hex()[:8]}...")
                    
                parent_hash = self.hash_fn(left + right)
                next_layer.append(parent_hash)
                
            self.layers.append(next_layer)
            current_layer = next_layer
            print(f"  [Layer {layer_idx + 1} Total Nodes: {len(current_layer)}]\n")
            layer_idx += 1

    def get_proof(self, index: int) -> list[tuple[bytes, bool]]:
        proof = []
        for layer in self.layers[:-1]:
            pair_idx = index + 1 if index % 2 == 0 else index - 1
            is_left_sibling = pair_idx < index
            
            if pair_idx < len(layer):
                proof.append((layer[pair_idx], is_left_sibling))
            else:
                proof.append((layer[index], not is_left_sibling))
                
            index //= 2
        return proof

    def verify_proof(self, leaf: bytes, proof: list[tuple[bytes, bool]], root: bytes) -> bool:
        current_hash = self.hash_fn(leaf)
        for sibling_hash, is_left in proof:
            if is_left:
                current_hash = self.hash_fn(sibling_hash + current_hash)
            else:
                current_hash = self.hash_fn(current_hash + sibling_hash)
        return current_hash == root

# ==========================================
# 3. RUNTIME EXECUTION
# ==========================================
transactions = [b"Tx0_Alpha", b"Tx1_Beta", b"Tx2_Gamma", b"Tx3_Delta", b"Tx4_Epsilon"]
target_idx = 2
target_leaf = transactions[target_idx]
fake_leaf = b"Tx2_Tampered_Data"

for name, hash_func in hash_variants.items():
    print("=" * 70)
    print(f" TESTING LAYERED GENERATION: {name} ".center(70, "#"))
    print("=" * 70)
    
    # 1. Build and display structural layer calculation
    tree = MerkleTree(transactions, hash_fn=hash_func)
    print(f"Final Merkle Root Generated (Hex): {tree.root.hex()}\n")
    
    # 2. Extract proof
    proof = tree.get_proof(target_idx)
    print(f"Generated Proof Path for Index {target_idx}:")
    for step, (h, side) in enumerate(proof):
        side_str = "Left" if side else "Right"
        print(f"  Layer {step} Sibling: {h.hex()[:16]}... (Position: {side_str})")
        
    # 3. Verify proofs
    is_valid = tree.verify_proof(target_leaf, proof, tree.root)
    is_fake_valid = tree.verify_proof(fake_leaf, proof, tree.root)
    
    print(f"\nIs the proof valid for pristine data? -> {is_valid}")
    print(f"Did verification safely catch tampered data? -> {not is_fake_valid}\n\n")

############ TESTING LAYERED GENERATION: Standard SHA-256 ############
  [Layer 0 -> Layer 1 Processing]
    - Hashing children pairs: 538b9c2a... + dd46a6bb...
    - Hashing children pairs: 3d249a2f... + 9f5211ab...
    - Odd child duplication: c991f975... + (Duplicated) c991f975...
  [Layer 1 Total Nodes: 3]

  [Layer 1 -> Layer 2 Processing]
    - Hashing children pairs: 373b9554... + d2a0dcee...
    - Odd child duplication: 3231d57b... + (Duplicated) 3231d57b...
  [Layer 2 Total Nodes: 2]

  [Layer 2 -> Layer 3 Processing]
    - Hashing children pairs: c934bdc2... + b214a69f...
  [Layer 3 Total Nodes: 1]

Final Merkle Root Generated (Hex): 798f14ed03615400e00418db5bdfc592243bb82865a095a3fa63bc71baf6e2df

Generated Proof Path for Index 2:
  Layer 0 Sibling: 9f5211ab1e9b0b13... (Position: Right)
  Layer 1 Sibling: 373b955412875523... (Position: Left)
  Layer 2 Sibling: b214a69f88558bca... (Position: Right)

Is the proof valid for pristine data? -> True
Did verification safely catch t

In [ ]:
import random
import time

p = 2**61 - 1
m = 3
c = 1
s = 128
output_len = 1

cpu_rp = RescuePrime(p, m , c, s, enable_gpu=False)
gpu_rp = RescuePrime(p, m, c, s, enable_gpu=True)

print("Generating Batched Merkle Tree Test")
batch_sz = 8192
elements_per_msg = 8
test_batch = [random.randint(0, 255) for _ in range(batch_sz)]

start_cpu = time.perf_counter()
cpu_tree = MerkleTree(test_batch, hash=cpu_rp.hash_batch, is_batched=True)
cpu_result = cpu_tree.root.hex()
cpu_time = time.perf_counter() - start_cpu
print ("CPU Batch Completed in: {cpu_time} seconds with hash {cpu_result}")

start_gpu = time.perf_counter()
gpu_tree = MerkleTree(test_batch, hash=gpu_rp.hash_batch, is_batched=True)
gpu_result = gpu_tree.root.hex()
gpu_time = time.perf_counter() - start_cpu
print ("GPU Batch Completed in: {cpu_time} seconds with hash {cpu_result}")
gpu_tree = MerkleTree (test_batch, hash=gpu_rp.hash_batch, is_batched=True)

if cpu_result == gpu_result:
    print("[PASS] Cryptographic Equivalence Maintained across all 10,000 hashes!")
    speedup = cpu_time / gpu_time
    print(f"[Speedup Factor]: GPU is {speedup:.2f}x FASTER than CPU.")
